# 线程层次结构

SIMT 采用 **Grid、Thread Block、Thread** 三级线程层次，从顶层到底层逐级划分并行任务。


## 1. 三级层次

<img style="display:block; margin-left:0; margin-right:auto;" src="./images/simt_thread_hierarchy.svg" alt="Grid、Thread Block 与 Thread 的层次关系" width="700px">

（图片来源：PyPTO 官方文档）

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">层级</th><th style="text-align:left;">构成</th><th style="text-align:left;">标识</th></tr>
<tr><td><b>Grid</b></td><td>多个 Thread Block</td><td>grid_dim = (grid_x, grid_y, grid_z)；各 Block 用自身坐标标识</td></tr>
<tr><td><b>Thread Block</b></td><td>若干 Thread</td><td>block_dim = (block_x, block_y, block_z)；块内 Thread 总数 ≤ 2048</td></tr>
<tr><td><b>Thread</b></td><td>最小编程单元</td><td>块内三维坐标（thread_idx）</td></tr>
</table>


## 2. Grid 与 Block 的规模从哪来

- **Grid（本课程的内嵌 SIMT VF 路径）**：每个执行到 SIMT 启动点的 **Vector 核启动一个 Thread Block**——grid_dim 的 X 维 = 外层 Kernel 实际生效的 Vector 核数（由 Host 侧 `block_dim` 间接决定）；当前仅使用 X 维，Y/Z 恒为 1；
- **Thread Block**：定义 SIMT 入口函数时声明 `max_threads`（单块上限），实际块尺寸由调用处 `simt_func[threads](...)` 的方括号决定；
- **线程总数** = grid_x × grid_y × grid_z × block_x × block_y × block_z。


## 3. dim3 三维结构

Grid 与 Block 的规模、Block 与 Thread 的坐标均为含 X/Y/Z 三个分量的 **dim3** 三维结构。方括号可写一至三个整数：`f[256]`、`f[16,16]`、`f[8,8,4]`——未给出的维度补 1，各维为编译期正整数，**乘积不得超过 max_threads 且不超过 2048**。


## 4. Warp：硬件的执行单位

**Warp 是硬件在线程块内组织执行的单位，Warp Size = 32**：线程按块内线性编号划分到 Warp；线程总数不是 32 的整数倍时，最后一个 Warp 只有部分线程有效。

同一 Warp 中的线程可以走不同分支（这正是 SIMT 的能力），但**分支发散会降低执行效率**——线程数取 32 的整数倍是可考虑的性能选择（详见 06.05）。


## 5. 线程到数据的映射：三种典型块形状

同样的 1024 个线程，块形状不同，映射方式就不同——这是 SIMT 设计的第一决策：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">块形状</th><th style="text-align:left;">调用写法</th><th style="text-align:left;">适合的数据</th><th style="text-align:left;">索引映射</th></tr>
<tr><td>一维 [1024]</td><td><code>f[1024](...)</code></td><td>向量 [1, N]</td><td><code>idx = bx * 1024 + tx</code></td></tr>
<tr><td>二维 [32, 32]</td><td><code>f[32, 32](...)</code></td><td>矩阵 [M, N] 块</td><td><code>row=task_row*32+ty, col=task_col*32+tx</code>（task 由一维 Block 编号及循环映射，不能依赖恒为 0 的 by）</td></tr>
<tr><td>三维 [8, 8, 16]</td><td><code>f[8, 8, 16](...)</code></td><td>多通道/批次</td><td>各维坐标独立，或展开为一维（linear_thread_idx）</td></tr>
</table>

**推演示例（一维，最常用）**：数据 [1, 4096] FP32、块 [256]、4 个 Vector 核生效：

```text
grid_dim().x = 4（每核一个 Block）
block_dim() = (256, 1, 1)

全局线程总数 = 4 × 256 = 1024
线程 t 的全局编号 = block_idx().x * 256 + thread_idx().x ∈ [0, 1024)
数据 4096 个 → 每线程 4 个元素 → 需要 grid-stride（第 7 章）：
    for i in range(全局编号, 4096, 1024): 处理 x[i]
```

**推演示例（二维）**：GM 矩阵 [128,128] 按块 [16,16] 处理，共 64 个块任务；多个实际 Block 跨步认领任务：

```text
for task in range(block_idx().x, 64, grid_dim().x):
    row = task // 8 * 16 + thread_idx().y
    col = task % 8 * 16 + thread_idx().x
    output[row, col] = ...  # GM 输出；各 Block 的 UB Tile 是各自私有的，不能拼成全局矩阵
```

要点：二维块的两个坐标**分开使用**（y 管行、x 管列），不要先展开成一维再除法拆回；一维展开也可用，选择时比较可读性与实际产码。


## 6. Warp 视角下的块形状选择

块内线程按 **X 维优先**编号划分 Warp：`f[16, 16]` 的前 32 个线程包含 y=0,1 两行的全部 x。**X 维优先不等于整个 Warp 访问连续地址**，还要看线程坐标如何映射到数据：

- 当 block.x=32，且 x 对应矩阵列时，一个 Warp 可沿同一行访问连续 32 个元素；访存合并效果仍取决于对齐等条件。
- 对 `f[16,16]`，若 `row=thread_idx().y, col=thread_idx().x`，同一 Warp 访问两段各 16 个连续元素。对于 [128,128] 行主序矩阵，两段起点相隔 128 个元素，不能把它们当作连续 32 个元素。
- 若把 thread_idx().x 映射到矩阵行、列保持不变，相邻线程的地址会相隔一整行；这是另一种跨行步长访问。

**设计习惯**：让"最快变化的线程维度（x）"对齐"最快变化的数据维度（最后一轴）"，与 SIMD 世界"最后一轴连续"的直觉一致。


## 7. SIMT 的硬件落位：跑在哪、数据怎么流动

线程模型之外，看一眼硬件（官方《SIMT 抽象硬件架构》）——理解 Shared Memory 与访存代价的出处：

<img style="display:block; margin-left:0; margin-right:auto;" src="./images/simt_abstract_hardware_architecture.svg" alt="SIMT 抽象硬件架构" width="650px">

（图片来源：PyPTO 官方文档）

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">资源</th><th style="text-align:left;">说明</th><th style="text-align:left;">与编程模型的对应</th></tr>
<tr><td>计算单元</td><td>执行 SIMT 函数的标量计算、地址计算、控制逻辑</td><td>函数体即计算</td></tr>
<tr><td>寄存器与栈空间</td><td>每线程独立；参数、线程索引、局部变量、中间结果</td><td>线程的"私有领地"</td></tr>
<tr><td><b>Shared Memory</b></td><td><b>UB 的部分空间</b>，同 Block 内线程交换数据与复用中间结果</td><td>第 6 章 06.04 的共享 Tile 载体</td></tr>
<tr><td>Data Cache</td><td>UB 的部分空间，缓存 SIMT 线程访问的 GM 数据</td><td>透明，但解释"重复读 GM 为什么快"</td></tr>
<tr><td>L2 Cache</td><td>GM 与各 Vector Core 之间，多 Core 共享、硬件管理</td><td>跨 Block 数据访问的中转</td></tr>
</table>

**一条贯穿的数据流**：SIMT 线程访问 GM → 经 L2 Cache 与 Vector Core 内 Data Cache → 进入线程私有寄存器。块内共享的数据放在 Shared Memory（UB 分区），由块内线程直接访问——这就是 06.04 "共享 Tile" 的硬件本貌，也是"共享 Tile 的跨线程读写需用 syncthreads 等同步保证可见性"的物理来源（写进的是片上共享区，不是线程私有寄存器）。

> 把本节与 06.04 连起来读：**线程层次（本节）→ 硬件落位（本节）→ 共享 Tile 与同步（06.04）**，SIMT 的世界观就完整了。


### 数一遍第 3 个 Block 的最后一个线程

4 个 Block、每块 256 线程时，block=3、thread=255 对应全局索引 3×256+255=1023。若有效数据只有 1000 个，block=3 中 thread=0～231 访问有效数据，thread=232～255 不访问。最后一块“有 232 个有效线程”不等于启动时只创建 232 个线程。

二维块 [16,16] 的线性编号是 y×16+x。编号 17 对应 (x=1,y=1)，编号 31 对应 (15,1)。这能解释同一个 Warp 横跨两行的来源。先写出坐标，再决定访问的是当前核私有 UB 还是全局 GM，避免把块内下标直接用于整幅矩阵。


## 8. 课后练习

1. （填空）Warp Size = ______；一个 100 线程的 Block 含 ______ 个 Warp，最后一个 Warp 只有 ______ 个有效线程。
2. （选择）`simt_func[8, 8, 4](...)` 的块内线程总数：
   A. 16<br/>B. 256<br/>C. 84
3. （判断）Pro 的 Grid 中，每个 Thread Block 由一个执行到 SIMT 启动点的 Vector 核启动。（对/错）

In [ ]:
!cat ./answer/06.02_answer.txt

## 9. 本节小结

- 三级层次：**Grid（Block 集）→ Thread Block（≤2048 线程）→ Thread**；
- Pro 的 Grid 特殊性：一个 Vector 核一个 Block，X 维 = 生效核数；
- dim3 三维，方括号 1~3 个编译期正整数；**Warp=32，线程数对齐可减少残缺 Warp，但不保证加速**。